# timeseries_toolkit — streaming bandpower demo (notebook)

This notebook is a **single-file**, easy-to-run walkthrough:

- Create a synthetic signal (burst + chirp + noise)
- Compute **offline reference** bandpower
- Compute **streaming (chunked)** bandpower (SOS IIR + EMA)
- Visualize and compare

Assumes you've run:

```bash
cd ml-play/timeseries/timeseries_toolkit
python -m pip install -e .
```


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from timeseries_toolkit.streaming import chunk_stream
from timeseries_toolkit.streaming.bandpower import SOSIIRBandPower
from timeseries_toolkit.eval import offline_sos_ema_bandpower

np.set_printoptions(precision=3, suppress=True)


## 1) Simple synthetic signal generator

This is an in-notebook generator (so you can run immediately). Later, you can move it to:
`src/timeseries_toolkit/signals/ground_truth.py`.


In [ ]:
def make_bursty_chirp_signal(fs=1000.0, T=10.0, f0=70.0, f1=150.0, burst_center_s=5.0, burst_width_s=1.2,
                            snr_db=0.0, rng=None):
    """Return (t, x_clean, x_noisy).
    
    - x_clean: chirp within a Gaussian burst envelope
    - x_noisy: x_clean + white noise set to target SNR (approx in power)
    """
    if rng is None:
        rng = np.random.default_rng(0)
    fs = float(fs)
    N = int(round(T * fs))
    t = np.arange(N) / fs

    # Linear chirp phase
    k = (f1 - f0) / max(T, 1e-9)
    phase = 2 * np.pi * (f0 * t + 0.5 * k * t**2)
    chirp = np.sin(phase)

    # Gaussian burst envelope
    env = np.exp(-0.5 * ((t - burst_center_s) / (burst_width_s / 2.355))**2)  # width_s ~ FWHM
    x_clean = env * chirp

    # Scale noise to achieve desired SNR in power
    sig_power = np.mean(x_clean**2)
    snr_lin = 10 ** (snr_db / 10)
    noise_power = sig_power / max(snr_lin, 1e-12)
    noise = rng.standard_normal(N) * np.sqrt(noise_power)

    x_noisy = x_clean + noise
    return t, x_clean, x_noisy


fs = 1000.0
t, x_clean, x = make_bursty_chirp_signal(fs=fs, T=10.0, snr_db=-3.0, rng=np.random.default_rng(1))

plt.figure()
plt.plot(t, x, label='x (noisy)', linewidth=1)
plt.plot(t, x_clean, label='x_clean', linewidth=2, alpha=0.8)
plt.title('Synthetic signal: bursty chirp + noise')
plt.xlabel('Time (s)')
plt.ylabel('Amplitude')
plt.legend()
plt.tight_layout()
plt.show()


## 2) Offline reference bandpower (SOS IIR + EMA)

This is the "ground truth" for our streaming estimator (same operations, just not chunked).


In [ ]:
band = (70.0, 150.0)
order = 4
tau_s = 0.2

bp_ref = offline_sos_ema_bandpower(x, fs=fs, band=band, order=order, ema_tau_s=tau_s)

plt.figure()
plt.plot(t, bp_ref)
plt.title('Offline reference bandpower')
plt.xlabel('Time (s)')
plt.ylabel('Bandpower (a.u.)')
plt.tight_layout()
plt.show()


## 3) Streaming / chunked bandpower

We process the signal in chunks and update the estimator state (filter state + EMA state).


In [ ]:
est = SOSIIRBandPower(fs=fs, band=band, order=order, ema_tau_s=tau_s)
chunk_size = 256

bp_stream_chunks = []
for ch in chunk_stream(x, chunk_size=chunk_size, hop_size=chunk_size, pad_end=False):
    bp_stream_chunks.append(est.update(ch))
bp_stream = np.concatenate(bp_stream_chunks)

print('ref shape:', bp_ref.shape, 'stream shape:', bp_stream.shape)


### Compare streaming vs offline


In [ ]:
Tmin = min(len(bp_ref), len(bp_stream))
err = bp_stream[:Tmin] - bp_ref[:Tmin]
print('max abs err:', float(np.max(np.abs(err))))
print('rmse:', float(np.sqrt(np.mean(err**2))))

plt.figure()
plt.plot(t[:Tmin], bp_ref[:Tmin], label='offline ref')
plt.plot(t[:Tmin], bp_stream[:Tmin], '--', label='streaming (chunked)')
plt.title('Bandpower: offline vs streaming')
plt.xlabel('Time (s)')
plt.ylabel('Bandpower (a.u.)')
plt.legend()
plt.tight_layout()
plt.show()


## 4) Overlay raw signal + bandpower

This makes it easy to see that the bandpower rises where the burst occurs.


In [ ]:
plt.figure()
plt.plot(t, x, label='signal x', linewidth=1)
plt.plot(t[:Tmin], (bp_ref[:Tmin] / (bp_ref[:Tmin].max() + 1e-12)) * np.std(x) * 2,
         label='bandpower (normalized)', linewidth=2)
plt.title('Signal + normalized bandpower overlay')
plt.xlabel('Time (s)')
plt.ylabel('Amplitude / scaled power')
plt.legend()
plt.tight_layout()
plt.show()


## 5) Multi-channel example (optional)

Here we create 3 channels with different mixtures of clean/noise and run the same estimator.


In [ ]:
rng = np.random.default_rng(2)
C = 3
X = np.vstack([
    x,
    0.5 * x_clean + rng.standard_normal(len(x)) * 0.4,
    0.2 * x_clean + rng.standard_normal(len(x)) * 0.8,
])

bp_ref_mc = offline_sos_ema_bandpower(X, fs=fs, band=band, order=order, ema_tau_s=tau_s)
est_mc = SOSIIRBandPower(fs=fs, band=band, order=order, ema_tau_s=tau_s)

bps = []
for ch in chunk_stream(X, chunk_size=256, hop_size=256, axis=-1, pad_end=False):
    bps.append(est_mc.update(ch))
bp_stream_mc = np.concatenate(bps, axis=-1)

Tmin = min(bp_ref_mc.shape[1], bp_stream_mc.shape[1])

plt.figure()
for c in range(C):
    plt.plot(t[:Tmin], bp_ref_mc[c, :Tmin], label=f'ch{c} ref')
plt.title('Offline reference bandpower (multi-channel)')
plt.xlabel('Time (s)')
plt.ylabel('Bandpower (a.u.)')
plt.legend()
plt.tight_layout()
plt.show()

plt.figure()
for c in range(C):
    plt.plot(t[:Tmin], bp_stream_mc[c, :Tmin], '--', label=f'ch{c} stream')
plt.title('Streaming bandpower (multi-channel)')
plt.xlabel('Time (s)')
plt.ylabel('Bandpower (a.u.)')
plt.legend()
plt.tight_layout()
plt.show()
